# NB-06: Code Quality & Pipeline Integrity Checker

Audits the benchmark codebase for:
- Duplicate case names in the 74-task catalogue
- Stale `hybrid_system_v40` imports (should be v50-2)
- Split protocol mismatches (PCA 40/60 vs random 80/20)
- Exposed API keys in notebooks
- **FIX-C3 result verification** — reads `exp2_pca_4060_summary.json`, `exp1_pca_summary.json`, and `fixc3_baseline.json` to confirm the corrected Feynman and DeFi PCA solve rates are present after C5c-0/1/2 complete.

**CI note:** All paths are resolved relative to this notebook's location — no `git clone` or `os.chdir()` is used. Place this notebook inside `notebooks/` at the repository root before executing.

## Setup — resolve repo root

In [ ]:
import collections
import glob
import json
import re
import sys
from pathlib import Path
import matplotlib
matplotlib.use("Agg")  # headless-safe backend; no display required
import matplotlib.pyplot as plt

# Notebook lives in notebooks/ — repo root is one level up.
SCRIPT_DIR = Path().resolve()
REPO_ROOT = SCRIPT_DIR.parent if (SCRIPT_DIR.parent / "hypatiax").exists() else SCRIPT_DIR

print(f"Repo root : {REPO_ROOT}")
print(f"Notebook  : {SCRIPT_DIR}")


## Step 1 — Duplicate case names in benchmark catalogue

In [ ]:
BENCH_FILE = REPO_ROOT / "hypatiax/experiments/benchmarks/hypatiax_defi_benchmark_v3c.py"

if not BENCH_FILE.exists():
    print(f"[NOT FOUND]  {BENCH_FILE}")
    print("Searching recursively...")
    candidates = list(REPO_ROOT.rglob("*defi_benchmark*.py"))
    print(f"Candidates: {[str(c) for c in candidates]}" if candidates else "  No candidates found.")
else:
    bench_src = BENCH_FILE.read_text(encoding="utf-8")
    print(f"Loaded: {BENCH_FILE}  |  {len(bench_src):,} chars")

    NAME_RE = re.compile(r'"name"\s*:\s*"([^"]+)"')
    names = NAME_RE.findall(bench_src)
    print(f"\nCase names found: {len(names)}")

    counter = collections.Counter(names)
    dupes = {k: v for k, v in counter.items() if v > 1}
    print(f"Duplicate case names: {len(dupes)}")
    for name, count in sorted(dupes.items()):
        print(f"  DUPLICATE ({count}x): {name!r}")

    if not dupes:
        print("  OK - no duplicate names found")


## Step 2 — Stale v40 import check

In [ ]:
FILES_TO_CHECK = [
    "hypatiax/experiments/benchmarks/hypatiax_defi_benchmark_v3c.py",
    "hypatiax/experiments/benchmarks/run_comparative_suite_benchmark_v2.py",
    "hypatiax/experiments/benchmarks/run_dual_condition_benchmark.py",
    "hypatiax/experiments/benchmarks/run_dual_sweep_benchmarks.py",
    "hypatiax/experiments/benchmarks/run_hybrid_system_benchmark.py",
    "hypatiax/experiments/benchmarks/run_noise_sweep_benchmark.py",
    "hypatiax/experiments/benchmarks/run_sample_complexity_benchmark.py",
]

print("Checking for stale hybrid_system_v40 imports:")
print("-" * 80)
for fname in FILES_TO_CHECK:
    fpath = REPO_ROOT / fname
    if not fpath.exists():
        print(f"  [NOT FOUND]  {fname}")
        continue
    src = fpath.read_text(encoding="utf-8")
    v40_hits = [(i+1, ln.strip()) for i, ln in enumerate(src.splitlines())
                if "hybrid_system_v40" in ln and not ln.strip().startswith("#")]
    v50_hits = [(i+1, ln.strip()) for i, ln in enumerate(src.splitlines())
                if "hybrid_system_v50_2" in ln or "hybrid_system_v5" in ln]
    if v40_hits:
        print(f"  [STALE v40]  {fname}")
        for lno, ctx in v40_hits[:3]:
            print(f"    line {lno}: {ctx[:80]}")
    elif v50_hits:
        print(f"  [OK - v50]   {fname}")
    else:
        print(f"  [NO IMPORT]  {fname}")


## Step 3 — Split protocol audit (PCA 40/60 vs random 80/20)

In [ ]:
SPLIT_FILES = [
    "hypatiax/experiments/benchmarks/hypatiax_defi_benchmark_v3c.py",
    # FIX-C3-SCRIPT: pca.py is the corrected Feynman runner (pca_directed_split, method-level)
    "hypatiax/experiments/benchmarks/run_comparative_suite_benchmark_pca.py",
    # v2.py retained for audit — its random 80/20 split is the LEGACY baseline
    "hypatiax/experiments/benchmarks/run_comparative_suite_benchmark_v2.py",
]

print("Split protocol audit:")
print("  Paper claims: PCA-directed 40%/60% aggressive extrapolation split")
print("-" * 80)

for fname in SPLIT_FILES:
    fpath = REPO_ROOT / fname
    if not fpath.exists():
        print(f"  [NOT FOUND]  {fname}")
        continue
    src = fpath.read_text(encoding="utf-8")
    has_pca  = "pca" in src.lower()
    has_8020 = "test_size=0.2" in src or "test_size=0.20" in src
    has_4060 = "0.4" in src and ("0.6" in src or "60" in src)

    pca_lines   = [(i+1, ln.strip()) for i, ln in enumerate(src.splitlines())
                   if "pca" in ln.lower() and "import" not in ln.lower()][:3]
    split_lines = [(i+1, ln.strip()) for i, ln in enumerate(src.splitlines())
                   if "train_test_split" in ln][:3]

    print(f"\n  File: {fname}")
    print(f"    Has PCA mention  : {has_pca}")
    print(f"    Has 80/20 split  : {has_8020}")
    print(f"    Has 40/60 split  : {has_4060}")
    for lno, ctx in pca_lines:
        print(f"    PCA line {lno}: {ctx[:80]}")
    for lno, ctx in split_lines:
        print(f"    Split line {lno}: {ctx[:80]}")

print()
print("STATUS:")
print("  run_comparative_suite_benchmark_v2.py: uses train_test_split(test_size=0.2)")
print("  (random 80/20) — this is the LEGACY baseline, locked in fixc3_baseline.json.")
print("  run_comparative_suite_benchmark_pca.py: uses pca_directed_split(test_size=0.6)")
print("  (PCA 40/60) — FIX-C3 corrected runner. Outputs in exp2_pca_4060/.")
print("  Gates A/B/C in ci_runner_disclosure.yml must PASS before results are valid.")


## Step 3b — FIX-C3 result verification

Reads the summary JSONs written by `run_all.sh` after C5c-0/1/2 complete.
Passes only when all three summary files exist and contain non-null solve rates.
This is the post-experiment check that complements Gates A/B/C (which verify
code structure) — this cell verifies **actual numerical output is present**.

In [ ]:
import json, os
from pathlib import Path

# Resolve RESULTS_DIR — prefer env var (set by CI), fall back to repo-relative default
RESULTS_DIR = Path(os.environ.get("RESULTS_DIR",
                   str(REPO_ROOT / "hypatiax/data/results")))

FIXC3_CHECKS = [
    (
        "exp2_feynman_pca_4060 (Feynman corrected)",
        RESULTS_DIR / "comparison_results/feynman-tests/exp2_pca_4060/exp2_pca_4060_summary.json",
        "solve_rate",
        "comparison_results/feynman-tests/exp2/",   # legacy dir for comparison
        RESULTS_DIR / "fixc3_baseline.json",
    ),
    (
        "exp1_pca (DeFi all-74 PCA)",
        RESULTS_DIR / "comparison_results/noise-noiseless/noiseless/defi_pca/exp1_pca_summary.json",
        "solve_rate",
        None,
        None,
    ),
]

print("FIX-C3 result verification")
print("-" * 80)

all_ok = True

for label, summary_path, rate_key, legacy_dir, baseline_path in FIXC3_CHECKS:
    print(f"\n  [{label}]")
    if not summary_path.exists():
        print(f"    [MISSING] {summary_path.name} not found")
        print(f"    -> Run the corresponding C5c step in ci_runner_disclosure.yml")
        all_ok = False
        continue

    try:
        data = json.loads(summary_path.read_text())
    except Exception as e:
        print(f"    [ERROR] Could not parse {summary_path.name}: {e}")
        all_ok = False
        continue

    rate      = data.get(rate_key)
    n_pass    = data.get("n_pass",  "?")
    n_total   = data.get("n_total", "?")
    protocol  = data.get("split_protocol", "?")

    if rate is None:
        print(f"    [FAIL] solve_rate is null — results not yet computed")
        all_ok = False
    else:
        print(f"    [OK]  solve_rate = {rate:.3f}  ({n_pass}/{n_total})  protocol={protocol!r}")

    # Show baseline comparison for Feynman
    if baseline_path and baseline_path.exists():
        try:
            bl = json.loads(baseline_path.read_text())
            bl_rate = bl.get("solve_rate")
            bl_claim = bl.get("paper_claim", "?")
            if bl_rate is not None and rate is not None:
                delta = rate - bl_rate
                direction = "higher" if delta > 0 else "lower" if delta < 0 else "same"
                print(f"    [OK]  Baseline (random 80/20): {bl_rate:.3f} (paper claim: {bl_claim})")
                print(f"    [OK]  Corrected vs baseline  : {delta:+.3f} ({direction}, as expected for harder split)")
        except Exception as e:
            print(f"    [WARN] Could not read baseline: {e}")
    elif baseline_path:
        print(f"    [MISSING] fixc3_baseline.json not found — Gate C must run first")
        all_ok = False

    # Check disclosure file
    disc = summary_path.parent / "split_protocol_disclosure.json"
    if disc.exists():
        try:
            dm = json.loads(disc.read_text())
            print(f"    [OK]  Disclosure: split_level={dm.get('split_level')!r}  "
                  f"force_fresh={dm.get('force_fresh')}")
        except Exception:
            print(f"    [WARN] Could not parse split_protocol_disclosure.json")
    else:
        print(f"    [WARN] split_protocol_disclosure.json missing — Gate B will flag this")

print()
if all_ok:
    print("FIX-C3 result verification: PASSED — all summary files present with non-null rates")
else:
    print("FIX-C3 result verification: INCOMPLETE — see MISSING/FAIL items above")
    print("  Re-run ci_runner_disclosure.yml after C5c experiments complete.")


## FIX-C3 (Option B applied) — `pca_directed_split` utility

The function below is the production implementation used by
`run_comparative_suite_benchmark_pca.py` (the dedicated FIX-C3 runner).
It is imported from `hypatiax/utils/pca_split_utils.py` at the top of that script
and called method-level inside `ImprovedNN.run()` — no CLI flags required.
`patch_benchmark_split.py` is superseded by the dedicated PCA script.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA

def pca_directed_split(X, y, test_size=0.6, random_state=None):
    """
    PCA-directed train/test split.

    Sorts samples along PC1 and splits at the (1 - test_size) quantile,
    producing an aggressive extrapolation scenario.

    Parameters
    ----------
    X : ndarray or DataFrame, shape (n_samples, n_features)
    y : ndarray or Series, length n_samples
    test_size : float, default 0.6
    random_state : int or None

    Returns
    -------
    X_train, X_test, y_train, y_test : ndarray
    """
    if not 0 < test_size < 1:
        raise ValueError(f"test_size must be between 0 and 1, got {test_size}")

    X_df = X.copy() if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
    y_s  = y.copy() if isinstance(y, pd.Series)    else pd.Series(y, name="target")

    n_samples, n_features = X_df.shape
    if n_samples == 0 or n_features == 0:
        raise ValueError("Cannot split an array with zero samples or features.")

    pc1 = PCA(n_components=min(n_samples, n_features, 1),
              random_state=random_state).fit_transform(X_df)[:, 0]

    order       = pd.Series(pc1, index=X_df.index).sort_values().index
    split_point = int(n_samples * (1.0 - test_size))
    train_idx, test_idx = order[:split_point], order[split_point:]

    X_train = X_df.loc[train_idx].values if isinstance(X, pd.DataFrame) else X[train_idx]
    X_test  = X_df.loc[test_idx].values  if isinstance(X, pd.DataFrame) else X[test_idx]
    y_train = y_s.loc[train_idx].values  if isinstance(y, pd.Series)    else y[train_idx]
    y_test  = y_s.loc[test_idx].values   if isinstance(y, pd.Series)    else y[test_idx]

    return X_train, X_test, y_train, y_test


### Smoke-test — synthetic data

In [ ]:
np.random.seed(42)
X_syn = np.vstack([
    np.random.normal(loc=[-5, -5], scale=1, size=(50, 2)),
    np.random.normal(loc=[ 5,  5], scale=1, size=(50, 2)),
])
y_syn = np.array([0]*50 + [1]*50)

X_tr, X_te, y_tr, y_te = pca_directed_split(X_syn, y_syn, test_size=0.6, random_state=42)
print(f"Train: {X_tr.shape}  Test: {X_te.shape}")
assert X_tr.shape == (40, 2) and X_te.shape == (60, 2), "Unexpected split sizes"
print("Split sizes OK")

# Visualise (headless-safe — Agg backend set in setup cell)
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(X_tr[:, 0], X_tr[:, 1], c=y_tr, cmap="viridis", label="Train (40%)")
ax.scatter(X_te[:, 0], X_te[:, 1], c=y_te, cmap="plasma",
           marker="x", s=80, label="Test (60%)")
ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
ax.set_title("PCA-Directed 40/60 Split — smoke test")
ax.legend(); ax.grid(True)
plt.tight_layout()
plt.savefig("pca_split_smoketest.png", dpi=80)
plt.close()
print("Plot saved to pca_split_smoketest.png")


## Step 4 — Exposed API key scan

In [ ]:
notebooks_found = list(REPO_ROOT.rglob("*.ipynb"))
print(f"Scanning {len(notebooks_found)} notebooks for exposed API keys...")
print("-" * 80)

KEY_RE = re.compile(r"sk-ant-api\d+-[A-Za-z0-9_-]{20,}")
found_any = False

for nb_path in notebooks_found:
    try:
        nb_data = json.loads(nb_path.read_text(encoding="utf-8"))
        for cell in nb_data.get("cells", []):
            src = "".join(cell.get("source", []))
            hits = KEY_RE.findall(src)
            if hits:
                found_any = True
                print(f"  [CRITICAL] EXPOSED API KEY in {nb_path.relative_to(REPO_ROOT)}")
                for m in hits:
                    print(f"    Key prefix: {m[:30]}...")
    except Exception as exc:
        print(f"  [ERROR] Could not read {nb_path}: {exc}")

if not found_any:
    print("  OK - no exposed API keys found in notebooks")
else:
    print()
    print("  ACTION: Rotate the exposed key at console.anthropic.com IMMEDIATELY.")
    print("  Remove the key from the notebook before any git commit or sharing.")


## Step 5 — Fix recipe summary

In [ ]:
print(
    "FIX-C3  Feynman split protocol mismatch (\u00a710.7) \u2014 RESOLVED\n"
    "  run_comparative_suite_benchmark_pca.py uses pca_directed_split(test_size=0.6)\n"
    "  (PCA 40/60 split at outer-loop scope), matching the DeFi benchmark (\u00a76.4).\n"
    "  Legacy 9/30 result (random 80/20) locked in fixc3_baseline.json.\n"
    "  Corrected Feynman results: comparison_results/feynman-tests/exp2_pca_4060/.\n"
    "  Corrected DeFi PCA results: comparison_results/noise-noiseless/noiseless/defi_pca/.\n"
    "  Gates A/B/C in ci_runner_disclosure.yml: run after every C5c trigger.\n"
    "  Step 3b above confirms summary files exist with non-null solve rates.\n"
    "\n"
    "  REMAINING ACTION: Update \u00a710.7 in paper to cite the corrected result\n"
    "  from exp2_pca_4060_summary.json (or explicitly disclose that the 9/30\n"
    "  baseline used a random 80/20 split while DeFi used PCA 40/60).\n"
    "  NB-04 TRUTH dict must also be updated: feynman_successes / feynman_total\n"
    "  should reflect the corrected solve rate once \u00a710.7 is revised.\n"
)
print("=" * 80)
print("NB-06 audit complete.")
